# RNN 주가 예측: 삼성전자와 SK하이닉스 📈

**한국외국어대학교 인공지능 · RNN 실습**

RNN은 순서가 있는 데이터를 한 시점씩 읽으면서 은닉상태 h<sub>t</sub>에 지금까지의 내용을 요약해 다음 시점으로 넘긴다. 지난 실습에서는 리뷰를 한 글자씩 읽었고, 이번에는 주가를 하루씩 읽는다. 삼성전자와 SK하이닉스의 최근 30거래일을 RNN에 넣어 다음 거래일을 예측하고, 그 예측을 단순한 기준선과 모의투자 결과에 비교한다. 예측 그래프가 그럴듯해 보이는 것과 예측이 실제로 쓸모 있는 것을 구분하는 방법을 익힌다.

| 단계 | 할 일 | 시간 |
|---|---|---|
| 1 | 삼성전자·SK하이닉스 일봉 받기 | 1분 |
| 2 | 미니 게임: 차트를 보고 다음 날 방향 맞히기 | 3분 |
| 3 | 30거래일 창 만들기, RNN 모델 정의 | 2분 |
| 4 | 실험 1: 내일 종가 맞히기, 「어제 종가 그대로」와 비교 | 5분 |
| 5 | 실험 2: 내일 오를까 내릴까, 기본 RNN·LSTM·GRU 비교 | 5분 |
| 6 | 실험 3: 간밤 미국 반도체 주가를 입력에 추가 | 5분 |
| 7 | 모의투자: AI 대 원숭이 1,000마리 | 5분 |
| 8 | **게임**: 사람 대 AI | 자유 |
| 9 | 다음 거래일 예측 출력 | 1분 |
| 10 | 보너스: 방향 대신 변동 크기 맞히기 | 2분 |

**시작 전에**: `런타임 → 모두 실행`. 모델이 작아서 GPU 없이도 몇 분 안에 끝난다. API 키는 필요 없다. 그래프의 제목과 축은 Colab에서 한글 글꼴이 깨지지 않도록 영어로 표시한다.

데이터: 삼성전자(005930)·SK하이닉스(000660) 일봉은 네이버 금융, 마이크론(MU) 일봉은 야후 파이낸스에서 [FinanceDataReader](https://github.com/FinanceData/FinanceDataReader)로 받는다. 실행할 때마다 전 거래일까지의 데이터를 새로 받으므로 숫자는 실행한 날에 따라 달라진다.

이 노트북은 수업용 실험이다. 결과를 투자 판단에 쓰지 않는다.

## 0. 준비

In [ ]:
import importlib.util, subprocess, sys, time, random
for pip_name, module in [("finance-datareader", "FinanceDataReader"), ("gradio", "gradio")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "-q", "install", pip_name], check=False)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
import torch, torch.nn as nn, torch.nn.functional as F
from IPython.display import display

# 그래프의 제목·축·범례는 Colab에서 한글 글꼴이 깨지지 않도록 영어로 표시한다.
NAVY, TEAL, RUST, GRAY = "#002B49", "#146E7A", "#A6432E", "#B7C4C8"
RED, BLUE = "#C8102E", "#1F5FBF"          # 한국 증시 표기: 상승은 빨강, 하락은 파랑

SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. 삼성전자·SK하이닉스 일봉 받기

일봉은 하루에 한 줄씩 기록한 주가다. 이 실습에서는 그날 처음 거래된 가격인 **시가(Open)**와 마지막 가격인 **종가(Close)**를 쓴다. 2015년부터 전 거래일까지를 받고, **2024년까지를 학습 구간, 2025년부터를 평가 구간**으로 나눈다.

In [ ]:
STOCKS = {"Samsung Electronics": "005930", "SK hynix": "000660"}      # 그래프에 쓸 영어 이름 : 종목코드
KOR = {"Samsung Electronics": "삼성전자", "SK hynix": "SK하이닉스"}
US_TICKER = "MU"            # 6단계에서 쓸 미국 반도체 주식: 마이크론
START = "2015-01-01"
SPLIT = "2025-01-01"        # 이 날짜 앞은 학습 구간, 뒤는 평가 구간
W = 30                      # 모델이 한 번에 보는 과거 거래일 수
COPY = "https://raw.githubusercontent.com/juho127/hufs-dl-practice/main/data/stock_{}.csv"

def load(code):
    # 일봉(시가 Open, 종가 Close)을 받는다. 네이버·야후에서 직접 받고, 둘 다 안 되면 저장소 사본을 쓴다.
    def from_fdr():
        import FinanceDataReader as fdr
        return fdr.DataReader(code, START)
    def from_yahoo():
        import yfinance as yf
        d = yf.Ticker(code + ".KS" if code.isdigit() else code).history(start=START, auto_adjust=False)
        d.index = d.index.tz_localize(None).normalize(); return d
    def from_copy():
        return pd.read_csv(COPY.format(code), index_col=0, parse_dates=True)
    for source, fn in [("FinanceDataReader", from_fdr), ("yfinance", from_yahoo), ("저장소 사본", from_copy)]:
        try:
            d = fn()[["Open", "Close"]].astype(float).dropna()
            if len(d) > 1000: return d, source
        except Exception:
            pass
    raise RuntimeError(f"{code} 데이터를 받지 못했다. 잠시 뒤 이 셀을 다시 실행한다.")

now = pd.Timestamp.now(tz="Asia/Seoul"); today = now.normalize().tz_localize(None)
frames = {}
for name, code in STOCKS.items():
    d, source = load(code)
    frames[name] = d[d.index < today]          # 오늘 장은 아직 끝나지 않았을 수 있으므로 전 거래일까지 쓴다
    print(f"{KOR[name]}({code}): {source}에서 {len(frames[name]):,}거래일")

common = sorted(set.intersection(*[set(d.index) for d in frames.values()]))
P = pd.DataFrame({n: d.loc[common, "Close"] for n, d in frames.items()})        # 종가
OPEN = pd.DataFrame({n: d.loc[common, "Open"] for n, d in frames.items()})      # 시가
OPEN = OPEN.where(OPEN > 0, P)                 # 거래정지일은 시가가 0으로 기록되어 있어 종가로 채운다
R = P.pct_change().fillna(0) * 100             # 하루 등락률(%): 전 거래일 종가 대비
dates = P.index
train_day = np.asarray(dates < SPLIT)          # 학습 구간에 속하는 날짜 표시

print(f"\n기간 {dates[0].date()} ~ {dates[-1].date()} · 거래일 {len(dates):,}일 = 학습 {train_day.sum():,}일 + 평가 {(~train_day).sum():,}일")
P.tail(5).rename(index=lambda d: d.date(), columns=KOR).style.format("{:,.0f}")

두 구간의 성격을 비교한다. 왼쪽 그래프는 첫날 가격을 100으로 맞춘 주가(세로축은 로그 눈금), 오른쪽은 하루 등락률이다. 회색으로 칠한 부분이 평가 구간이다.

In [ ]:
rows = []
for name in STOCKS:
    for label, mask in [("학습 (~2024)", train_day), ("평가 (2025~)", ~train_day)]:
        r = R[name].values[mask]
        rows.append({"종목": KOR[name], "구간": label, "최저 종가(원)": P[name].values[mask].min(), "최고 종가(원)": P[name].values[mask].max(),
                     "하루 등락률의 표준편차(%)": r.std(), "오른 날의 비율(%)": (r > 0).mean() * 100,
                     "가장 크게 오른 날(%)": r.max(), "가장 크게 내린 날(%)": r.min()})
display(pd.DataFrame(rows).set_index(["종목", "구간"]).style.format("{:,.1f}").format("{:,.0f}", subset=["최저 종가(원)", "최고 종가(원)"]))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
for name, color in zip(STOCKS, [NAVY, RUST]):
    axes[0].plot(dates, P[name] / P[name].iloc[0] * 100, color=color, lw=1.4, label=name)
    axes[1].plot(dates, R[name], color=color, lw=0.6, alpha=0.75, label=name)
axes[0].set_yscale("log"); axes[0].set_title("Close price (first day = 100, log scale)"); axes[0].legend()
axes[1].set_title("Daily change (%)"); axes[1].legend()
for ax in axes: ax.axvspan(pd.Timestamp(SPLIT), dates[-1], color=GRAY, alpha=0.3); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

> 🤔 평가 구간은 학습 구간과 무엇이 다른가? 최고 종가와 하루 등락률의 표준편차를 비교해 보자. 모델은 2024년까지의 주가만 보고 학습한 뒤, 한 번도 본 적 없는 가격대와 등락 폭을 만나게 된다. 이 차이가 4단계에서 문제가 된다.

### 🕵️ 2. 미니 게임: 차트를 보고 다음 날 방향 맞히기

AI를 만들기 전에 사람이 먼저 해 본다. 아래 8개 차트는 평가 구간에서 뽑은 30거래일의 주가다(첫날 = 100, 종목과 날짜는 가렸다). 각 차트의 **다음 거래일 종가가 전날보다 오를지(↑) 내릴지(↓)** 적어 보자. 다음 셀에서 정답이 나온다.

In [ ]:
quiz_rng = np.random.default_rng(2026)
candidates = [(n, t) for n in STOCKS for t in np.flatnonzero(~train_day) if R[n].values[t] != 0]
quiz = [candidates[i] for i in quiz_rng.choice(len(candidates), 8, replace=False)]     # (종목, 정답 날짜의 위치) 8개

def quiz_plot(reveal):
    fig, axes = plt.subplots(2, 4, figsize=(13, 5))
    for q, (ax, (name, t)) in enumerate(zip(axes.ravel(), quiz), 1):
        c = P[name].values; path = c[t - W:t] / c[t - W] * 100
        ax.plot(range(1, W + 1), path, color=NAVY, lw=1.8)
        title = f"Q{q}"
        if reveal:
            up = c[t] > c[t - 1]
            ax.plot([W, W + 1], [path[-1], c[t] / c[t - W] * 100], "o-", color=RED if up else BLUE, lw=2.2)
            title += f": {'UP' if up else 'DOWN'} {R[name].values[t]:+.1f}%"
        ax.set_title(title, fontsize=11); ax.set_xticks([1, 10, 20, 30]); ax.grid(alpha=0.25)
    fig.supxlabel("trading day"); fig.supylabel("price (day 1 = 100)")
    plt.tight_layout(); plt.show()

quiz_plot(reveal=False)

In [ ]:
quiz_plot(reveal=True)
for q, (name, t) in enumerate(quiz, 1):
    print(f"Q{q}. {'상승' if R[name].values[t] > 0 else '하락'} {R[name].values[t]:+.1f}%  ({KOR[name]}, {dates[t].date()})")

> 🤔 8문제 중 몇 개를 맞혔는가? 동전을 던져도 평균 4개는 맞힌다. 자기 점수를 적어 두고 5단계와 6단계에서 AI의 점수와 비교한다.

## 3. 30거래일 창 만들기

RNN에 넣을 수 있도록 데이터를 **창(window)**으로 자른다. 창 하나는 연속한 30거래일의 입력이고, 정답은 그 다음 거래일의 값이다. 창을 하루씩 옆으로 밀면서 학습 예제를 만든다.

```
입력: 30거래일 × 특성 수 → 순환층(RNN, LSTM, GRU 중 하나, 은닉 32) → 마지막 은닉상태 h_T → 선형층 → 다음 거래일 예측값 1개
```

30일을 다 읽은 뒤 **마지막 은닉상태 하나로 답을 내는 다대일** 형태다. 강의자료 M7의 「생각해 볼 질문」(최근 30일의 주가로 다음 날 주가를 예측하는 모델)과 같은 설정이다.

시계열을 다룰 때 지켜야 할 규칙이 두 가지 있다.

- **시간 순서로 나눈다.** 창을 무작위로 섞어 학습·평가로 나누면 평가일보다 뒤의 주가가 학습에 들어간다. 여기서는 정답 날짜가 2025년 이전이면 학습, 이후면 평가에 쓴다.
- **정규화 기준은 학습 구간에서만 구한다.** 평균, 표준편차, 최고가를 전체 기간에서 구하면 미래의 정보를 미리 쓰는 것이다.

In [ ]:
def make_windows(inputs, target):
    # inputs: (거래일 수, 특성 수) 배열, target: (거래일 수,) 배열
    # X[i]는 정답 날짜 직전 W거래일의 입력, y[i]는 정답 날짜의 값이다.
    X = np.stack([inputs[t - W:t] for t in range(W, len(inputs))]).astype(np.float32)
    y = np.asarray(target[W:], dtype=np.float32)
    return X, y

def standardize(a):
    # 학습 구간의 평균과 표준편차로 표준화한다. 평가 구간의 통계는 쓰지 않는다.
    return (a - a[train_day].mean()) / a[train_day].std()

target_dates = dates[W:]                              # 창마다의 정답 날짜
TR = np.asarray(target_dates < SPLIT); TE = ~TR       # 학습용 창, 평가용 창
test_pos = np.arange(W, len(dates))[TE]               # 평가용 창의 정답 날짜 위치

first = list(STOCKS)[0]
X, y = make_windows(R[[first]].values, R[first].values)
print(f"창 {len(X):,}개 = 학습 {TR.sum():,}개 + 평가 {TE.sum():,}개 · 창 하나의 모양 {X.shape[1:]} (30거래일 × 특성 1개)")
print(f"첫 번째 창: {dates[0].date()} ~ {dates[W - 1].date()}의 등락률 → 정답: {dates[W].date()}의 등락률 {y[0]:+.2f}%")

In [ ]:
class StockRNN(nn.Module):
    def __init__(self, cell="LSTM", n_in=1, hid=32):
        super().__init__()
        self.rnn = getattr(nn, cell)(n_in, hid, batch_first=True)      # nn.RNN, nn.LSTM, nn.GRU 가운데 하나
        self.head = nn.Linear(hid, 1)
    def forward(self, x):                              # x: (묶음 크기, 30거래일, 특성 수)
        out, _ = self.rnn(x)                           # out: 30거래일 각각의 은닉상태 h_t
        return self.head(out[:, -1]).squeeze(1)        # 마지막 날의 은닉상태 h_T 하나로 예측한다 (다대일)

def train_model(cell, X, y, loss_fn, epochs=20, seed=SEED):
    torch.manual_seed(seed)
    model = StockRNN(cell, X.shape[2]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    X, y = torch.tensor(X, device=device), torch.tensor(y, device=device)
    for ep in range(epochs):
        model.train(); order = torch.randperm(len(X), device=device)
        for i in range(0, len(order), 128):
            b = order[i:i + 128]
            loss = loss_fn(model(X[b]), y[b])
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    return model

@torch.no_grad()
def predict(model, X):
    model.eval()
    return model(torch.tensor(X, device=device)).cpu().numpy()

rmse = lambda a, b: float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))
for cell in ["RNN", "LSTM", "GRU"]:
    print(f"{cell:4s} 파라미터 {sum(p.numel() for p in StockRNN(cell).parameters()):,}개")

> 🤔 같은 은닉 크기 32에서 LSTM의 파라미터가 기본 RNN의 약 4배, GRU가 약 3배인 이유는 무엇인가? 강의에서 본 게이트의 개수와 연결해 보자.

## 4. 실험 1: 내일 종가 맞히기

### 4-1. 가격을 그대로 넣기

가장 먼저 떠오르는 방법이다. 최근 30거래일의 종가를 넣고 다음 거래일의 종가를 맞히게 한다. 가격은 학습 구간의 최저가를 0, 최고가를 1로 맞춰서 넣는다.

In [ ]:
fig, axes = plt.subplots(1, len(STOCKS), figsize=(6.5 * len(STOCKS), 3.6))
for ax, name in zip(np.atleast_1d(axes), STOCKS):
    c = P[name].values
    lo, hi = c[train_day].min(), c[train_day].max()            # 학습 구간의 최저가, 최고가
    s = (c - lo) / (hi - lo)                                   # 학습 구간 기준으로 0~1에 맞춘다
    X, y = make_windows(s[:, None], s)
    model = train_model("LSTM", X[TR], y[TR], F.mse_loss)
    pred = predict(model, X[TE]) * (hi - lo) + lo              # 다시 원 단위로
    actual = c[W:][TE]
    print(f"{KOR[name]}: 학습 구간 최고가 {hi:,.0f}원 · 평가 구간 최고가 {actual.max():,.0f}원 · LSTM 예측의 최고값 {pred.max():,.0f}원 · RMSE {rmse(pred, actual):,.0f}원")
    ax.plot(target_dates[TE], actual, color=NAVY, lw=1.5, label="actual close")
    ax.plot(target_dates[TE], pred, color=RUST, lw=1.5, label="LSTM prediction")
    ax.axhline(hi, color=GRAY, ls="--", label="highest price in training period")
    ax.set_title(f"{name}: predicting the price itself"); ax.legend(fontsize=8); ax.grid(alpha=0.25); ax.tick_params(axis="x", labelrotation=30)
plt.tight_layout(); plt.show()

> 🤔 예측선이 실제 주가를 따라가지 못하는 구간은 어디인가? 회색 점선은 학습 구간의 최고가다. 모델은 그보다 높은 가격을 입력으로도 정답으로도 본 적이 없다. 학습 때 본 범위를 벗어난 입력에서는 신경망의 출력을 믿기 어렵다.

### 4-2. 가격 대신 등락률로 바꾸기

가격대가 달라져도 쓸 수 있도록 입력과 정답을 **전 거래일 대비 등락률(%)**로 바꾼다. 주가가 5만 원이든 30만 원이든 +2%는 같은 +2%다. 예측한 등락률을 전날 종가에 곱하면 예측 종가가 된다.

In [ ]:
ret_pred = {}
fig, axes = plt.subplots(1, len(STOCKS), figsize=(6.5 * len(STOCKS), 3.6))
for ax, name in zip(np.atleast_1d(axes), STOCKS):
    r = R[name].values
    X, y = make_windows(standardize(r)[:, None], r)            # 입력: 과거 30거래일 등락률, 정답: 다음 거래일 등락률(%)
    model = train_model("LSTM", X[TR], y[TR], F.mse_loss)
    ret_pred[name] = predict(model, X[TE])
    c = P[name].values; actual, prev = c[W:][TE], c[W - 1:-1][TE]          # 정답 날짜의 종가, 그 전날 종가
    pred_price = prev * (1 + ret_pred[name] / 100)
    print(f"{KOR[name]}: RMSE {rmse(pred_price, actual):,.0f}원 · 실제 종가와의 상관계수 {np.corrcoef(pred_price, actual)[0, 1]:.4f}")
    ax.plot(target_dates[TE], actual, color=NAVY, lw=1.5, label="actual close")
    ax.plot(target_dates[TE], pred_price, color=TEAL, lw=1.2, label="LSTM prediction")
    ax.set_title(f"{name}: predicting the daily change"); ax.legend(fontsize=8); ax.grid(alpha=0.25); ax.tick_params(axis="x", labelrotation=30)
plt.tight_layout(); plt.show()

이번에는 예측선이 실제 주가와 거의 겹친다. 인터넷에서 흔히 보는 「LSTM 주가 예측」 그래프가 이런 모습이다. 이 그래프만 보고 예측에 성공했다고 말해도 될까?

### 4-3. 「어제 종가 그대로」와 비교하기

아무것도 학습하지 않는 기준선을 하나 만든다. **내일 종가는 오늘 종가와 같다**고 답하는 방법이다. 모델의 오차를 이 기준선의 오차와 비교한다.

In [ ]:
rows = []
fig, axes = plt.subplots(2, len(STOCKS), figsize=(6.5 * len(STOCKS), 7))
axes = axes.reshape(2, -1)
for j, name in enumerate(STOCKS):
    c = P[name].values; actual, prev = c[W:][TE], c[W - 1:-1][TE]
    pred_price = prev * (1 + ret_pred[name] / 100); actual_ret = R[name].values[W:][TE]
    rows.append({"종목": KOR[name], "LSTM의 RMSE(원)": rmse(pred_price, actual), "「어제 종가 그대로」의 RMSE(원)": rmse(prev, actual),
                 "예측 등락률의 표준편차(%)": ret_pred[name].std(), "실제 등락률의 표준편차(%)": actual_ret.std(),
                 "예측·실제 등락률의 상관계수": np.corrcoef(ret_pred[name], actual_ret)[0, 1]})
    d = target_dates[TE][-30:]                                  # 마지막 30거래일을 확대한다
    axes[0, j].plot(d, actual[-30:], "o-", color=NAVY, lw=1.6, ms=3, label="actual close")
    axes[0, j].plot(d, pred_price[-30:], "o-", color=TEAL, lw=1.4, ms=3, label="LSTM prediction")
    axes[0, j].plot(d, prev[-30:], "--", color=RUST, lw=1.4, label="yesterday's close")
    axes[0, j].set_title(f"{name}: last 30 trading days"); axes[0, j].legend(fontsize=8); axes[0, j].grid(alpha=0.25); axes[0, j].tick_params(axis="x", labelrotation=30)
    axes[1, j].scatter(actual_ret, ret_pred[name], s=8, color=TEAL, alpha=0.5)
    lim = np.abs(actual_ret).max() * 1.05
    axes[1, j].plot([-lim, lim], [-lim, lim], color=GRAY, ls="--", label="perfect prediction")
    axes[1, j].set_xlim(-lim, lim); axes[1, j].set_ylim(-lim, lim); axes[1, j].set_xlabel("actual change (%)"); axes[1, j].set_ylabel("predicted change (%)")
    axes[1, j].set_title(f"{name}: predicted vs actual daily change"); axes[1, j].legend(fontsize=8); axes[1, j].grid(alpha=0.25)
display(pd.DataFrame(rows).set_index("종목").style.format("{:,.2f}").format("{:,.0f}", subset=["LSTM의 RMSE(원)", "「어제 종가 그대로」의 RMSE(원)"]))
plt.tight_layout(); plt.show()

> 🤔 LSTM의 RMSE는 「어제 종가 그대로」의 RMSE보다 얼마나 작은가? 위쪽 확대 그래프에서 LSTM 예측선(청록)은 실제 주가(남색)와 어제 종가(주황 점선) 중 어느 쪽에 붙어 있는가? 아래쪽 산점도에서 점들이 회색 점선을 따라 늘어서 있는가?

4-2의 그래프가 그럴듯했던 이유를 확인했다. 모델은 「내일은 오늘과 거의 같다」고 답했고, 주가는 하루에 몇 % 안에서 움직이므로 멀리서 보면 두 선이 겹친다. 예측 모델을 평가할 때는 그래프의 모양이 아니라 **아무것도 학습하지 않는 기준선보다 얼마나 나은지**를 본다.

## 5. 실험 2: 내일 오를까 내릴까

질문을 바꾼다. 종가의 값이 아니라 **다음 거래일 종가가 전날보다 오르는지(1) 아닌지(0)**를 맞힌다. 출력은 상승 확률이고, 이진 분류이므로 손실은 교차 엔트로피를 쓴다. 등락률이 0인 보합은 0으로 둔다.

순환층만 바꿔서 기본 RNN, LSTM, GRU를 비교한다. 입력은 두 가지를 시험한다.

- `past`: 그 종목의 과거 30거래일 등락률
- `pair`: 여기에 다른 종목의 등락률을 추가 (삼성전자를 예측할 때 SK하이닉스도 함께 본다)

같은 설정에서도 가중치의 초깃값에 따라 결과가 달라지므로 **시드를 바꿔 3번씩** 학습하고 평균, 최저, 최고를 함께 본다.

In [ ]:
CELLS, SEEDS = ["RNN", "LSTM", "GRU"], [0, 1, 2]
UP = {name: (R[name].values > 0).astype(np.float32) for name in STOCKS}       # 그날 올랐으면 1

def inputs_for(name, kind):
    cols = [standardize(R[name].values)]                                       # past: 자기 등락률
    if kind == "pair":  cols += [standardize(R[o].values) for o in STOCKS if o != name]     # 다른 종목의 등락률
    if kind == "night": cols += [standardize(AFTER)]                           # 간밤 미국 주식 등락률 (6단계에서 만든다)
    return np.stack(cols, 1)

clf, prob, dir_rows = {}, {}, []
def run_direction(kind):
    t0 = time.time()
    for name in STOCKS:
        X, y = make_windows(inputs_for(name, kind), UP[name])
        for cell in CELLS:
            for seed in SEEDS:
                m = train_model(cell, X[TR], y[TR], F.binary_cross_entropy_with_logits, seed=seed)
                z_tr, z_te = predict(m, X[TR]), predict(m, X[TE])              # 0보다 크면 상승 쪽
                clf[name, kind, cell, seed] = m
                prob[name, kind, cell, seed] = 1 / (1 + np.exp(-z_te))         # 평가 구간의 상승 확률
                dir_rows.append({"종목": KOR[name], "입력": kind, "모델": cell, "seed": seed,
                                 "학습 정확도": ((z_tr > 0) == (y[TR] > 0.5)).mean() * 100, "평가 정확도": ((z_te > 0) == (y[TE] > 0.5)).mean() * 100})
    print(f"입력 {kind}: 모델 {len(STOCKS) * len(CELLS) * len(SEEDS)}개 학습 {time.time() - t0:.0f}초")

def show_direction(kinds):
    df = pd.DataFrame(dir_rows); df = df[df["입력"].isin(kinds)]
    table = df.groupby(["종목", "입력", "모델"], sort=False).agg(**{"학습 정확도(평균)": ("학습 정확도", "mean"), "평가 정확도(평균)": ("평가 정확도", "mean"),
                                                                 "평가 최저": ("평가 정확도", "min"), "평가 최고": ("평가 정확도", "max")})
    display(table.style.format("{:.1f}%"))
    fig, axes = plt.subplots(1, len(STOCKS), figsize=(6.5 * len(STOCKS), 3.6), sharey=True)
    width = 0.8 / len(kinds)
    for ax, name in zip(np.atleast_1d(axes), STOCKS):
        for i, (kind, color) in enumerate(zip(kinds, [GRAY, TEAL, RUST])):
            t = df[(df["종목"] == KOR[name]) & (df["입력"] == kind)].groupby("모델", sort=False)["평가 정확도"].agg(["mean", "min", "max"])
            x = np.arange(len(CELLS)) + (i - (len(kinds) - 1) / 2) * width
            ax.bar(x, t["mean"], width * 0.92, color=color, label=f"input: {kind}")
            ax.errorbar(x, t["mean"], yerr=[t["mean"] - t["min"], t["max"] - t["mean"]], fmt="none", ecolor="black", capsize=3)     # 시드 3개의 최저~최고
        base = UP[name][W:][TE].mean() * 100
        ax.axhline(50, color="black", ls="--", lw=1, label="coin flip (50%)"); ax.axhline(base, color=NAVY, ls=":", lw=1.5, label=f"always 'up' ({base:.1f}%)")
        ax.set_xticks(range(len(CELLS))); ax.set_xticklabels(CELLS); ax.set_ylim(40, 78); ax.set_title(f"{name}: test accuracy (%), bars = mean of 3 seeds"); ax.legend(fontsize=8, ncol=3, loc="upper center"); ax.grid(alpha=0.25, axis="y")
    plt.tight_layout(); plt.show()

run_direction("past"); run_direction("pair")
print("\n기준선 (평가 구간)")
for name in STOCKS:
    y, yesterday = UP[name][W:][TE], UP[name][W - 1:-1][TE]
    print(f"  {KOR[name]}: 항상 '오른다'고 답하면 {y.mean():.1%} · 어제와 같은 방향이라고 답하면 {(yesterday == y).mean():.1%}")
show_direction(["past", "pair"])

In [ ]:
def ensemble(name, kind):          # LSTM 3개(시드 3개)의 상승 확률 평균
    return np.mean([prob[name, kind, "LSTM", s] for s in SEEDS], 0)
def quiz_score(kind):              # 2단계 미니 게임 8문제를 AI가 풀었을 때의 점수
    return sum(int((ensemble(n, kind)[t - test_pos[0]] > 0.5) == (R[n].values[t] > 0)) for n, t in quiz)
print(f"미니 게임 8문제 · 차트만 본 LSTM의 점수: {quiz_score('past')} / 8")

> 🤔 세 모델의 평가 정확도는 동전 던지기(50%)와 얼마나 다른가? 항상 「오른다」고 답하는 기준선(남색 점선)보다 높은가? 학습 정확도와 평가 정확도의 차이는 무엇을 뜻하는가? 다른 종목을 입력에 추가하면(pair) 나아지는가?

기본 RNN을 LSTM이나 GRU로 바꿔도, 다른 종목을 함께 넣어도 평가 정확도는 50% 근처에 머문다. 학습 정확도가 평가 정확도보다 높다면 모델이 학습 구간의 우연한 등락을 외운 것이다(과적합). 과거 등락률에 내일의 방향을 알려 주는 규칙이 뚜렷하게 있다면 수많은 투자자가 이미 그 규칙대로 사고팔았을 것이고, 그 과정에서 규칙은 가격에 반영되어 사라진다.

막대 위의 검은 선은 시드 3개의 최저와 최고다. 시드 하나만 돌려서 나온 53%를 「성공」이라고 말하면 안 되는 이유가 여기에 있다.

## 6. 실험 3: 간밤 미국 반도체 주가를 입력에 추가

모델 구조를 바꿔서는 나아지지 않았다. 이번에는 **입력 정보**를 바꾼다. 삼성전자와 SK하이닉스는 메모리 반도체 회사이고, 같은 업종의 미국 회사 마이크론(MU)은 한국 장이 끝난 뒤에 열리는 미국 장에서 거래된다.

```
한국 t-1일 장 마감(15:30) → 미국 장(한국 시간 23:30~06:00, 서머타임에는 22:30~05:00) → 한국 t일 개장(09:00) → 한국 t일 장 마감
```

한국 t일 아침에는 간밤 마이크론의 등락률을 이미 알고 있다. 이 값을 입력에 추가한다. 창의 날짜마다 「그날 한국 장 마감 뒤 밤사이에 확정된 마이크론 등락률」을 붙이므로, 창의 마지막 줄에는 정답 날짜 아침에 알 수 있는 값이 들어간다. 정답인 한국 t일 종가는 그보다 뒤에 정해지므로 미래의 값을 미리 쓰는 것은 아니다.

In [ ]:
us, source = load(US_TICKER)
us = us["Close"]
us = us[us.index < (today - pd.Timedelta(days=1) if now.hour < 7 else today)]      # 아직 끝나지 않은 미국 장은 뺀다
print(f"{US_TICKER}: {source}에서 {len(us):,}거래일, 마지막 날 {us.index[-1].date()}")

# 미국 d일 종가는 한국 시간 d+1일 새벽에 확정된다.
# known[t]: 한국 거래일 t의 개장 전에 알 수 있는 가장 최근 미국 종가
known = us.copy(); known.index = known.index + pd.Timedelta(days=1)
known = known.reindex(dates.union(known.index)).ffill().reindex(dates)
# AFTER[j]: 한국 j일 장 마감 뒤, 다음 한국 거래일 개장 전까지 새로 확정된 미국 등락률(%)
after = known.shift(-1) / known - 1
after.iloc[-1] = us.iloc[-1] / known.iloc[-1] - 1          # 마지막 날에는 지금 알 수 있는 가장 최근 미국 종가를 쓴다
AFTER = (after.fillna(0) * 100).values

print("\n간밤 마이크론 등락률과 그날 한국 종목 등락률의 관계")
for name in STOCKS:
    night, kr = AFTER[:-1], R[name].values[1:]            # 간밤 미국 등락률, 그 뒤에 열린 한국 장의 등락률
    for label, mask in [("학습 구간", train_day[1:]), ("평가 구간", ~train_day[1:])]:
        print(f"  {KOR[name]} {label}: 상관계수 {np.corrcoef(night[mask], kr[mask])[0, 1]:.2f} · 방향이 같았던 날 {((night[mask] > 0) == (kr[mask] > 0)).mean():.1%}")

run_direction("night")
show_direction(["past", "pair", "night"])
print(f"미니 게임 8문제 · 차트만 본 LSTM: {quiz_score('past')} / 8 · 간밤 미국 주가도 본 LSTM: {quiz_score('night')} / 8")

> 🤔 입력에 간밤 마이크론 등락률 한 줄을 추가했을 때 평가 정확도는 얼마나 달라졌는가? 5단계에서 기본 RNN을 LSTM, GRU로 바꿨을 때의 변화와 비교해 보자.

같은 모델, 같은 학습 방법인데 입력 정보 하나로 결과가 달라졌다. 모델 구조를 고르는 것보다 **어떤 정보를 넣는가**가 결과를 더 크게 바꿀 때가 많다.

그러면 이 모델로 돈을 벌 수 있을까? 7단계에서 확인한다.

## 7. 💰 모의투자: AI 대 원숭이 1,000마리

평가 구간의 첫날에 1,000만 원을 들고 시작한다. 규칙은 단순하다. AI가 「내일 오른다」(상승 확률 50% 초과)고 하면 주식을 들고, 아니면 현금으로 둔다. AI는 LSTM 3개(시드 3개)의 상승 확률 평균을 쓴다.

비교 대상은 두 가지다.

- **계속 보유**: 첫날 사서 끝까지 들고 있는다.
- **원숭이 1,000마리**: 매일 동전을 던져 앞면이면 주식을 들고 뒷면이면 현금으로 둔다. 버턴 말킬의 『랜덤워크 투자수업』에 나오는, 눈을 가린 원숭이가 다트를 던져 고른 종목이 전문가가 고른 종목만큼 성과를 낸다는 이야기에서 가져온 무작위 기준선이다.

### 7-1. 차트만 본 AI

5단계의 `past` 모델이다. 과거 등락률은 전날 장 마감 때 모두 알 수 있으므로 **전날 종가에 사고판다**고 계산한다.

In [ ]:
MONEY = 1000          # 시작 금액(만 원)
COST = 0.0            # 사거나 팔 때마다 내는 비용(%): 도전 과제에서 0.2로 바꿔 본다
N_MONKEY = 1000

def grow(pos, daily, round_trip=False):
    # pos: 그날 주식을 들고 있으면 1, 현금이면 0 · daily: 그날의 수익률(소수) → 날마다의 평가 금액(만 원)
    pos = np.asarray(pos, dtype=float)
    trades = 2 * pos if round_trip else np.abs(np.diff(pos, axis=-1, prepend=0))     # 사거나 판 횟수
    return MONEY * np.cumprod((1 + daily * pos) * (1 - COST / 100) ** trades, axis=-1)

monkey_rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(len(STOCKS), 2, figsize=(13, 3.6 * len(STOCKS)))
axes = axes.reshape(len(STOCKS), 2)
for i, name in enumerate(STOCKS):
    cc = R[name].values[W:][TE] / 100                          # 전날 종가 → 당일 종가 수익률
    hold = grow(np.ones(len(cc)), cc)
    ai_pos = ensemble(name, "past") > 0.5
    ai = grow(ai_pos, cc)
    monkeys = grow(monkey_rng.random((N_MONKEY, len(cc))) < 0.5, cc)
    final = monkeys[:, -1]
    print(f"{KOR[name]}: 계속 보유 {hold[-1]:,.0f}만 원 · AI(차트만) {ai[-1]:,.0f}만 원 (주식을 든 날 {ai_pos.mean():.0%}) · "
          f"원숭이 중앙값 {np.median(final):,.0f}만 원 · AI는 원숭이 {N_MONKEY:,}마리와 함께 줄을 세우면 {(final > ai[-1]).sum() + 1:,}등")
    d = target_dates[TE]
    for m in monkeys[:60]: axes[i, 0].plot(d, m, color=GRAY, lw=0.5, alpha=0.6)
    axes[i, 0].plot(d, hold, color=NAVY, lw=2, label="buy and hold"); axes[i, 0].plot(d, ai, color=TEAL, lw=2, label="AI (chart only)")
    axes[i, 0].plot([], [], color=GRAY, label="60 of the monkeys")
    axes[i, 0].set_yscale("log"); axes[i, 0].set_title(f"{name}: account value (10,000 KRW, log scale)"); axes[i, 0].legend(fontsize=8); axes[i, 0].grid(alpha=0.25); axes[i, 0].tick_params(axis="x", labelrotation=30)
    axes[i, 1].hist(final, bins=40, color=GRAY)
    axes[i, 1].axvline(hold[-1], color=NAVY, lw=2, label="buy and hold"); axes[i, 1].axvline(ai[-1], color=TEAL, lw=2, label="AI (chart only)")
    axes[i, 1].set_title(f"{name}: final value of {N_MONKEY:,} monkeys"); axes[i, 1].set_xlabel("final account value (10,000 KRW)"); axes[i, 1].legend(fontsize=8); axes[i, 1].grid(alpha=0.25)
plt.tight_layout(); plt.show()

> 🤔 차트만 본 AI는 계속 보유보다 나았는가? 원숭이 1,000마리 가운데 몇 등인가? 원숭이들 사이의 최종 금액 차이는 어디에서 오는가?

원숭이는 아무 정보 없이 동전만 던졌는데도 최종 금액이 크게 갈린다. 운만으로 이만큼 차이가 난다는 뜻이다. 평가 구간처럼 주가가 크게 오른 시기에는 주식을 오래 든 쪽이 유리하므로, 한 번의 평가 구간에서 원숭이 다수를 이겼다는 결과만으로 실력이 있다고 말하기는 어렵다.

### 7-2. 간밤 미국 주가도 본 AI

6단계의 `night` 모델은 방향을 더 잘 맞혔다. 같은 방식으로 **전날 종가에 사고판다**고 계산해 본다.

In [ ]:
night_result = {}
for name in STOCKS:
    cc = R[name].values[W:][TE] / 100
    pos = ensemble(name, "night") > 0.5
    night_result[name] = dict(pos=pos, cc=cc, hold=grow(np.ones(len(cc)), cc), ai_close=grow(pos, cc))
    print(f"{KOR[name]}: 계속 보유 {night_result[name]['hold'][-1]:,.0f}만 원 · AI(간밤 정보), 전날 종가에 매매한 것으로 계산 {night_result[name]['ai_close'][-1]:,.0f}만 원")

계속 보유의 몇 배가 되는 금액이 나온다. 이 계산에는 잘못된 곳이 있다. 6단계의 시간표를 다시 보고 찾아보자.

모델이 쓰는 간밤 마이크론 등락률은 **한국 t일 아침**에야 알 수 있다. 위 계산은 그 정보를 보고 나서 **t-1일 종가**에 샀다고 가정했다. 이미 지나간 가격에는 살 수 없다. 실제로 살 수 있는 가장 빠른 가격은 **t일 시가**다. 그래서 「t일 시가에 사서 t일 종가에 판다」로 다시 계산하고, 하루의 등락을 두 구간으로 나눠서 본다.

- 전날 종가 → 당일 시가: 밤사이의 소식이 반영되는 구간
- 당일 시가 → 당일 종가: 실제로 사고팔 수 있는 구간

In [ ]:
rows = []
fig, axes = plt.subplots(1, len(STOCKS), figsize=(6.5 * len(STOCKS), 3.8))
for ax, name in zip(np.atleast_1d(axes), STOCKS):
    res = night_result[name]; pos = res["pos"]
    gap = (OPEN[name].values[1:] / P[name].values[:-1] - 1)[W - 1:][TE]       # 전날 종가 → 당일 시가
    oc = (P[name].values / OPEN[name].values - 1)[W:][TE]                     # 당일 시가 → 당일 종가
    ai_open = grow(pos, oc, round_trip=True)                                  # 오른다고 한 날만 시가에 사서 종가에 판다
    for label, mask in [("AI가 '오른다'고 한 날", pos), ("AI가 '오르지 않는다'고 한 날", ~pos)]:
        rows.append({"종목": KOR[name], "구분": label, "날 수": int(mask.sum()), "전날 종가→당일 종가(%)": res["cc"][mask].mean() * 100,
                     "전날 종가→당일 시가(%)": gap[mask].mean() * 100, "당일 시가→당일 종가(%)": oc[mask].mean() * 100})
    print(f"{KOR[name]}: 계속 보유 {res['hold'][-1]:,.0f}만 원 · 전날 종가 매매로 계산(불가능) {res['ai_close'][-1]:,.0f}만 원 · 당일 시가에 사서 종가에 팔기(가능) {ai_open[-1]:,.0f}만 원")
    d = target_dates[TE]
    ax.plot(d, res["ai_close"], color=RUST, lw=1.8, ls="--", label="AI + overnight info, trade at yesterday's close (impossible)")
    ax.plot(d, res["hold"], color=NAVY, lw=2, label="buy and hold")
    ax.plot(d, ai_open, color=TEAL, lw=2, label="AI + overnight info, buy at open, sell at close")
    ax.set_yscale("log"); ax.set_title(f"{name}: account value (10,000 KRW, log scale)"); ax.legend(fontsize=7.5); ax.grid(alpha=0.25); ax.tick_params(axis="x", labelrotation=30)
display(pd.DataFrame(rows).set_index(["종목", "구분"]).style.format("{:+.2f}").format("{:,.0f}", subset=["날 수"]))
plt.tight_layout(); plt.show()

> 🤔 AI가 「오른다」고 한 날의 평균 상승분은 두 구간 중 어디에 들어 있는가? 시가에 사서 종가에 파는 실제 가능한 매매의 결과는 계속 보유와 비교해 어떤가?

모델의 예측은 틀리지 않았다. 간밤에 마이크론이 오른 날에는 한국 반도체 주가도 전날 종가보다 높게 끝나는 경우가 많다. 그러나 그 상승분은 대부분 장이 열리는 순간의 시가에 이미 들어가 있다. 모두가 같은 소식을 보고 개장 전에 주문을 넣기 때문이다. **맞힐 수 있다는 것과 그 예측으로 수익을 낼 수 있다는 것은 다른 문제다.**

AI로 예측 모델을 만들 때 확인할 점을 정리하면 다음과 같다.

1. 아무것도 학습하지 않는 기준선과 비교했는가 (4단계)
2. 시드 하나의 결과가 아닌가 (5단계)
3. 입력 정보를 **실제로 알 수 있는 시점**과 그 예측으로 **행동할 수 있는 시점**이 맞는가 (7단계)

## 8. 🎮 게임: 사람 대 AI

아래 셀을 실행하면 게임 화면이 나온다. 평가 구간에서 무작위로 뽑은 30거래일 차트를 보고 다음 거래일의 방향을 맞힌다. 같은 문제를 AI 둘과 동전도 함께 푼다.

- **나**: 차트만 보고 답한다. 「힌트 보기」를 켜면 다음 문제부터 간밤 마이크론 등락률이 함께 나온다.
- **LSTM(차트만)**: 5단계의 `past` 모델
- **LSTM(차트+간밤 미국)**: 6단계의 `night` 모델
- **동전**: 무작위

먼저 힌트 없이 20문제를 풀어 점수를 적고, 힌트를 켜고 20문제를 더 풀어 비교해 보자.

In [ ]:
import gradio as gr
PLAYERS = ["나", "LSTM(차트만)", "LSTM(차트+간밤 미국)", "동전"]
P_UP = {(name, kind): ensemble(name, kind) for name in STOCKS for kind in ["past", "night"]}

def chart(name, t, reveal):
    c = P[name].values; path = c[t - W:t] / c[t - W] * 100
    fig = Figure(figsize=(6.4, 3.3)); ax = fig.subplots()
    ax.plot(range(1, W + 1), path, color=NAVY, lw=2)
    if reveal:
        ax.plot([W, W + 1], [path[-1], c[t] / c[t - W] * 100], "o-", color=RED if c[t] > c[t - 1] else BLUE, lw=2.5)
    ax.set_xlim(0, W + 2); ax.set_xlabel("trading day"); ax.set_ylabel("price (day 1 = 100)"); ax.grid(alpha=0.25)
    ax.set_title("Answer (red = up, blue = down)" if reveal else "Next trading day: up or down?")
    fig.tight_layout()
    return fig

def new_question(state, hint):
    while True:
        name, t = random.choice(list(STOCKS)), int(random.choice(test_pos))
        if R[name].values[t] != 0: break                          # 보합인 날은 문제로 내지 않는다
    state = dict(state, name=name, t=t, open=True)
    msg = "30거래일 차트다. 다음 거래일 종가는 전날보다 오를까, 내릴까?"
    if hint: msg += f"\n\n힌트: 간밤 마이크론 **{AFTER[t - 1]:+.1f}%**"
    return chart(name, t, False), msg, state

def answer(pick_up, state):
    if not state.get("open"):                                     # 이미 답한 문제는 다시 채점하지 않는다
        return gr.update(), gr.update(), gr.update(), state
    name, t = state["name"], state["t"]; k = t - test_pos[0]
    truth = bool(R[name].values[t] > 0)
    picks = [pick_up, bool(P_UP[name, "past"][k] > 0.5), bool(P_UP[name, "night"][k] > 0.5), random.random() < 0.5]
    score = [s + int(p == truth) for s, p in zip(state["score"], picks)]
    state = dict(state, open=False, n=state["n"] + 1, score=score)
    mark = lambda p: ("오른다" if p else "내린다") + (" ○" if p == truth else " ×")
    msg = (f"**정답: {'상승' if truth else '하락'} {R[name].values[t]:+.1f}%** · {KOR[name]} {dates[t].date()} · 간밤 마이크론 {AFTER[t - 1]:+.1f}%\n\n"
           + " · ".join(f"{who}: {mark(p)}" for who, p in zip(PLAYERS, picks)))
    board = "점수 · " + " | ".join(f"{who} **{s}/{state['n']}**" for who, s in zip(PLAYERS, score))
    return chart(name, t, True), msg, board, state

with gr.Blocks(title="사람 대 AI: 다음 날 주가 맞히기") as app:
    gr.Markdown("### 사람 대 AI: 30거래일 차트를 보고 다음 거래일의 방향을 맞힌다")
    state = gr.State({"n": 0, "score": [0, 0, 0, 0], "open": False})
    with gr.Row():
        with gr.Column(scale=3):
            plot = gr.Plot(label="차트 (첫날 = 100)")
        with gr.Column(scale=2):
            msg = gr.Markdown()
            with gr.Row():
                up_btn = gr.Button("📈 오른다", variant="primary"); down_btn = gr.Button("📉 내린다", variant="primary")
            next_btn = gr.Button("다음 문제")
            hint = gr.Checkbox(label="힌트 보기: 간밤 마이크론 등락률 (다음 문제부터 적용)", value=False)
            board = gr.Markdown("점수 · 아직 푼 문제가 없다")
    up_btn.click(lambda s: answer(True, s), [state], [plot, msg, board, state])
    down_btn.click(lambda s: answer(False, s), [state], [plot, msg, board, state])
    next_btn.click(new_question, [state, hint], [plot, msg, state])
    app.load(new_question, [state, hint], [plot, msg, state])
app.launch(debug=False)

> 🤔 힌트 없이 풀었을 때 내 점수는 차트만 본 LSTM, 동전과 얼마나 달랐는가? 힌트를 켠 뒤에는 어떻게 달라졌는가? 사람과 AI 모두 차트의 모양보다 간밤의 정보에서 도움을 받았다면, 차트 모양에 대해 어떤 결론을 내릴 수 있는가?

## 9. 다음 거래일 예측 출력

지금까지 학습한 방향 예측 모델 전부(순환층 3종 × 시드 3개)에 가장 최근 30거래일을 넣어 다음 거래일의 상승 확률을 출력한다. 장이 끝난 뒤 실제 결과와 비교해 보자. 각 모델 종류의 평가 정확도를 옆에 함께 적었다. 한 번 맞거나 틀린 것으로 모델을 평가할 수 없다는 점도 기억하자.

In [ ]:
accuracy = pd.DataFrame(dir_rows).groupby("입력")["평가 정확도"].mean()
print(f"데이터의 마지막 거래일: {dates[-1].date()} · 예측 대상: 그 다음 거래일의 종가가 {dates[-1].date()} 종가보다 오르는가")
print(f"간밤 마이크론 등락률: {AFTER[-1]:+.1f}% (미국 {us.index[-1].date()} 종가 기준)\n")
for name in STOCKS:
    print(f"{KOR[name]} · {dates[-1].date()} 종가 {P[name].iloc[-1]:,.0f}원")
    for kind, label in [("past", "차트만 본 모델"), ("night", "간밤 미국 주가도 본 모델")]:
        x = inputs_for(name, kind)[-W:][None].astype(np.float32)             # 가장 최근 30거래일
        p = np.array([1 / (1 + np.exp(-predict(clf[name, kind, cell, seed], x)[0])) for cell in CELLS for seed in SEEDS])
        print(f"   {label} {len(p)}개: 상승 확률 평균 {p.mean():.0%} · '오른다' 쪽 {(p > 0.5).sum()}개 · 이 종류 모델의 평가 정확도 평균 {accuracy[kind]:.1f}%")

## 10. 보너스: 방향 대신 변동 크기 맞히기

내일 오를지 내릴지는 맞히기 어려웠다. 질문을 한 번 더 바꿔서 **내일 얼마나 크게 움직일지**, 즉 등락률의 절댓값을 맞혀 본다. 주가는 크게 움직인 날 뒤에 크게 움직이는 날이 이어지는 경향이 있다(변동성 군집). 1단계의 등락률 그래프에서도 큰 등락이 몰려 있는 구간이 보인다.

기준선은 두 가지다. 학습 구간의 평균 변동 크기를 항상 답하는 방법, 그리고 최근 30거래일의 평균 변동 크기를 답하는 방법이다.

In [ ]:
rows = []
fig, axes = plt.subplots(1, len(STOCKS), figsize=(6.5 * len(STOCKS), 3.6))
for ax, name in zip(np.atleast_1d(axes), STOCKS):
    a = np.abs(R[name].values)                                              # 하루 등락률의 절댓값(%)
    X, y = make_windows(np.stack([standardize(R[name].values), a / a[train_day].mean()], 1), a)
    model = train_model("LSTM", X[TR], y[TR], F.mse_loss)
    pred, actual = predict(model, X[TE]), y[TE]
    recent = pd.Series(a).rolling(W).mean().values[W - 1:-1][TE]             # 최근 30거래일의 평균 변동 크기
    const = np.full(len(actual), a[train_day].mean())                       # 학습 구간의 평균 변동 크기
    for label, p in [("학습 구간 평균", const), ("최근 30거래일 평균", recent), ("LSTM", pred)]:
        rows.append({"종목": KOR[name], "방법": label, "평균 절대 오차(%p)": np.abs(p - actual).mean(),
                     "실제 변동 크기와의 상관계수": np.nan if np.ptp(p) < 1e-9 else np.corrcoef(p, actual)[0, 1]})
    d = target_dates[TE]
    ax.bar(d, actual, width=1.0, color=GRAY, label="actual |daily change|")
    ax.plot(d, recent, color=RUST, lw=1.5, label="mean of last 30 days"); ax.plot(d, pred, color=TEAL, lw=1.5, label="LSTM prediction")
    ax.set_title(f"{name}: size of the daily move (%)"); ax.legend(fontsize=8); ax.grid(alpha=0.25); ax.tick_params(axis="x", labelrotation=30)
display(pd.DataFrame(rows).set_index(["종목", "방법"]).style.format("{:.2f}", na_rep="-"))
plt.tight_layout(); plt.show()

> 🤔 LSTM의 예측과 실제 변동 크기의 상관계수는 4-3에서 본 등락률의 상관계수와 얼마나 다른가? LSTM은 「최근 30거래일 평균」이라는 단순한 방법보다 나은가?

방향과 달리 변동 크기에는 과거에서 읽을 수 있는 정보가 있다. 다만 최근 평균이라는 단순한 방법도 그 정보를 상당 부분 쓴다. 변동 크기의 예측은 손실 위험을 관리하고 옵션 가격을 정하는 데 쓰인다.

## 11. 도전 과제

1. **사람 대 AI 대회**: 8단계 게임을 힌트 없이 20문제, 힌트를 켜고 20문제 풀어 점수를 비교한다. 힌트가 있을 때 사람은 LSTM(차트+간밤 미국)을 이길 수 있는가?
2. **수수료 넣기**: 7단계의 `COST`를 0.2로 바꾸고 7단계 셀들을 다시 실행한다. 자주 사고파는 쪽과 계속 보유 중 어느 쪽이 더 크게 영향을 받는가?
3. **미래 정보 누출 체험**: 4-1에서 `lo, hi`를 `c[train_day]`가 아닌 `c` 전체에서 구하면 그래프가 어떻게 달라지는가? 결과가 좋아 보여도 이 방법을 쓰면 안 되는 이유를 설명한다.
4. **창 길이 바꾸기**: 1단계의 `W`를 5, 60, 120으로 바꿔 전체를 다시 실행한다. 창이 길어질 때 기본 RNN과 LSTM의 차이가 커지는가? 강의에서 본 기울기 소실과 연결해 본다.
5. **종목 바꾸기**: `STOCKS`와 `KOR`에 다른 종목을 넣는다(예: 현대차 `005380`, NAVER `035420`). 반도체가 아닌 종목에서도 마이크론이 도움이 되는가? `US_TICKER`를 그 업종의 미국 주식으로 바꾸면 어떤가?
6. **시가에서 종가까지의 방향 맞히기**: 5단계의 정답 `UP`을 「그날 종가가 시가보다 높은가」로 바꾼다(`P[name].values > OPEN[name].values`). 간밤 정보를 넣어도 정확도가 60%대로 오르는가? 7-2의 결과와 연결해 설명한다.
7. **다대다로 바꾸기**: 지금은 마지막 날의 은닉상태로만 오차를 계산한다. 30거래일 각각에서 다음 날을 맞히게 하면 학습 신호가 어떻게 달라지는가? (강의자료 M7의 「생각해 볼 질문」)

*데이터: 네이버 금융, 야후 파이낸스(FinanceDataReader). 이 노트북의 예측과 모의투자 결과는 수업용 실험이며 투자 권유가 아니다. 모의투자는 세금과 수수료, 원하는 가격에 체결되지 않을 가능성을 반영하지 않았다.*